# Index · Unit 7 — Evaluation & observability

Same shape as Units 1–6. For every component in **Section 8 of `index.md`**, two cells:

- **From scratch** — the mechanism by hand, no framework. Runs on plain Python /
  scripted *fake model* (no API key needed).
- **SOTA** — how it's actually done today (eval frameworks, LLM judges, tracing stacks).
  Some need `OPENAI_API_KEY`.

Each component lists **📚 Resources** (also consolidated in `RESOURCES.md`).

Components: **1** evals (outcome + trajectory) · **2** LLM-as-judge ·
**3** tracing / observability · **4** monitoring & feedback loops · **5** benchmarks.

**Intuition for the whole unit:** you can't improve what you can't measure, and
agents fail in ways a single "looks right" can't catch. Evaluation asks *did it get
the right answer, the right way?* (outcome **and** trajectory); observability asks
*what actually happened inside the run?* (every model/tool call, token, latency).
Together they turn a black box into something you can debug and trust. Same
pattern: small harnesses around the agent, which SOTA tools package.

## Setup

In [ ]:
import os, re, json

MODEL = "gpt-4o-mini"

print("✅ key set — SOTA cells will run" if os.environ.get("OPENAI_API_KEY")
      else "⚠️  no OPENAI_API_KEY — from-scratch cells still run; SOTA cells need a key")

def chat(messages, model=MODEL, **kwargs):
    from openai import OpenAI
    return OpenAI().chat.completions.create(
        model=model, messages=messages, **kwargs).choices[0].message

## 1. Evals — outcome + trajectory

An eval is a **test set for an agent**: fixed inputs with known-good outcomes, plus
a **scorer**. The twist vs. normal tests: you grade *two* things — the **outcome**
(final answer correct?) and the **trajectory** (did it take sensible steps — call
the right tool, not loop forever?). An agent can get the right answer by luck or a
wrong answer after perfect reasoning; you want to see both.

**From scratch:** a labelled dataset + an outcome scorer + a trajectory check.

> 📚 **Resources:** [OpenAI Evals](https://github.com/openai/evals) ·
> [promptfoo](https://www.promptfoo.dev/) · [LangSmith evaluation](https://docs.smith.langchain.com/evaluation)

In [ ]:
# From scratch — score outcome AND trajectory over a tiny dataset.
DATASET = [
    {"q": "2+2", "answer": "4", "expected_tool": "calculator"},
    {"q": "capital of France", "answer": "Paris", "expected_tool": "search"},
]

def fake_agent(q):
    # returns (answer, trajectory of tool calls)
    if re.search(r"\d", q): return "4", ["calculator"]
    return "Paris", ["search", "search"]        # took an extra, redundant step

def evaluate(dataset, agent):
    rows = []
    for ex in dataset:
        ans, traj = agent(ex["q"])
        outcome = ans.strip().lower() == ex["answer"].lower()
        traj_ok = ex["expected_tool"] in traj and len(traj) <= 1    # right tool, no waste
        rows.append({"q": ex["q"], "outcome": outcome, "trajectory_ok": traj_ok, "steps": len(traj)})
    acc  = sum(r["outcome"] for r in rows) / len(rows)
    teff = sum(r["trajectory_ok"] for r in rows) / len(rows)
    return {"outcome_accuracy": acc, "trajectory_score": teff, "rows": rows}

print(json.dumps(evaluate(DATASET, fake_agent), indent=2))

In [ ]:
# SOTA — semantic outcome scoring (exact match is too brittle for free text).
def semantic_match(got, expected):
    m = chat([
        {"role": "system", "content": "Does the ANSWER match the EXPECTED meaning? "
         'Reply JSON {"match": true/false}.'},
        {"role": "user", "content": f"EXPECTED: {expected}\nANSWER: {got}"},
    ], response_format={"type": "json_object"})
    return json.loads(m.content)["match"]

print("exact :", "The capital is Paris." == "Paris")                 # False, too strict
print("semant:", semantic_match("The capital is Paris.", "Paris"))   # True

## 2. LLM-as-judge

When there's no single correct string (summaries, essays, tone), grade with a
**model + rubric**. Two common forms: **pointwise** (score one output 1–5 on
criteria) and **pairwise** (which of A/B is better — more reliable, dodges
absolute-scale drift). Watch the known biases: position, verbosity, self-preference.

**From scratch:** a rubric scorer (fake judge) scoring on keyword criteria.

> 📚 **Resources:** [Judging LLM-as-a-Judge / MT-Bench — Zheng et al., 2023](https://arxiv.org/abs/2306.05685) ·
> [Anthropic — building evals & LLM graders](https://docs.anthropic.com/en/docs/test-and-evaluate/develop-tests)

In [ ]:
# From scratch — a rubric-based fake judge (pointwise).
RUBRIC = {"mentions_cost": ["cost", "price", "$"],
          "mentions_safety": ["safe", "guardrail", "risk"],
          "is_concise": None}       # None => special rule below

def judge(answer):
    a = answer.lower()
    scores = {}
    for crit, kws in RUBRIC.items():
        if kws is None:
            scores[crit] = len(answer.split()) <= 25         # concise = <=25 words
        else:
            scores[crit] = any(k in a for k in kws)
    total = sum(scores.values())
    return {"criteria": scores, "score": total, "max": len(RUBRIC)}

print(judge("A cheap model lowers cost but you must add safety guardrails."))
print(judge("It depends on many factors across the whole system and context."))

In [ ]:
# SOTA — a model judge with an explicit rubric + structured score (+ pairwise).
def llm_judge(question, answer):
    schema = {"type": "object", "properties": {
        "relevance": {"type": "integer"}, "correctness": {"type": "integer"},
        "clarity": {"type": "integer"}, "rationale": {"type": "string"}},
        "required": ["relevance", "correctness", "clarity", "rationale"],
        "additionalProperties": False}
    m = chat([
        {"role": "system", "content": "Grade the answer 1-5 on relevance, correctness, clarity. "
         "Be a strict, calibrated judge."},
        {"role": "user", "content": f"Q: {question}\nA: {answer}"},
    ], response_format={"type":"json_schema","json_schema":{"name":"grade","schema":schema,"strict":True}})
    return json.loads(m.content)

def pairwise(question, a, b):
    m = chat([{"role": "system", "content": "Which answer is better? Reply only 'A' or 'B'."},
              {"role": "user", "content": f"Q: {question}\nA: {a}\nB: {b}"}])
    return m.content.strip()

print(llm_judge("What is RAG?", "RAG retrieves documents and conditions generation on them."))
print("better:", pairwise("What is 2+2?", "4", "I think it is around four-ish"))

## 3. Tracing / logging / observability

To debug an agent you need the **whole run**, not just the answer: a tree of
**spans** — each model call, tool call, with inputs, outputs, tokens, latency,
nested by causality. That trace is how you find the step that went wrong, the tool
that was slow, or where the tokens went.

**From scratch:** a tiny tracer (context manager) recording nested spans into a
list.

> 📚 **Resources:** [OpenTelemetry GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/) ·
> [Langfuse](https://langfuse.com/) · [Arize Phoenix](https://github.com/Arize-ai/phoenix) ·
> [LangSmith tracing](https://docs.smith.langchain.com/observability)

In [ ]:
# From scratch — a nested-span tracer via a context manager.
from contextlib import contextmanager

TRACE = []
_stack = []

@contextmanager
def span(name, **attrs):
    rec = {"name": name, "attrs": attrs, "children": []}
    (_stack[-1]["children"] if _stack else TRACE).append(rec)
    _stack.append(rec)
    try:
        yield rec
    finally:
        _stack.pop()

# simulate an agent run
with span("agent_run", task="What is 2+2?"):
    with span("llm_call", model="gpt-4o-mini", tokens=42):
        pass
    with span("tool_call", tool="calculator", input="2+2", output="4"):
        pass
    with span("llm_call", model="gpt-4o-mini", tokens=18):
        pass

def show(nodes, depth=0):
    for n in nodes:
        print("  "*depth + f"• {n['name']} {n['attrs']}")
        show(n["children"], depth+1)
show(TRACE)

In [ ]:
# SOTA — real token/latency usage straight from the API response (feed to Langfuse/OTel).
def traced_call(prompt):
    import time
    from openai import OpenAI
    t0 = time.perf_counter()
    r = OpenAI().chat.completions.create(model=MODEL, messages=[{"role":"user","content":prompt}])
    span = {"model": MODEL,
            "latency_ms": round((time.perf_counter()-t0)*1000),
            "prompt_tokens": r.usage.prompt_tokens,
            "completion_tokens": r.usage.completion_tokens,
            "output": r.choices[0].message.content}
    return span   # in prod: langfuse.trace(...) / OpenTelemetry span with these attrs

print(json.dumps(traced_call("Say hi in one word."), indent=2))

## 4. Monitoring & feedback loops

Evals run offline on a fixed set; **monitoring** watches *production* over time:
success rate, cost, latency, plus **drift** (quality quietly degrading) and **user
feedback** (thumbs up/down) fed back into the eval set. The loop: log runs →
aggregate metrics → detect regressions → add failures to the eval set → improve.

**From scratch:** aggregate metrics over a batch and a simple drift check comparing
two time windows.

> 📚 **Resources:** [Langfuse — analytics & online eval](https://langfuse.com/docs) ·
> [Evidently AI — drift detection](https://github.com/evidentlyai/evidently)

In [ ]:
# From scratch — aggregate production metrics + a windowed drift check.
RUNS = [  # pretend production log (older -> newer)
    {"ok": True,  "cost": 0.01, "ms": 400}, {"ok": True,  "cost": 0.02, "ms": 420},
    {"ok": True,  "cost": 0.01, "ms": 410}, {"ok": True,  "cost": 0.02, "ms": 450},
    {"ok": False, "cost": 0.03, "ms": 900}, {"ok": False, "cost": 0.02, "ms": 950},
    {"ok": True,  "cost": 0.02, "ms": 980}, {"ok": False, "cost": 0.03, "ms": 1000},
]
def metrics(runs):
    n = len(runs)
    return {"success_rate": round(sum(r["ok"] for r in runs)/n, 2),
            "avg_cost": round(sum(r["cost"] for r in runs)/n, 3),
            "avg_ms": round(sum(r["ms"] for r in runs)/n)}

half = len(RUNS)//2
old, new = metrics(RUNS[:half]), metrics(RUNS[half:])
print("older window:", old)
print("newer window:", new)
if new["success_rate"] < old["success_rate"] - 0.1:
    print("🚨 DRIFT: success rate dropped", old["success_rate"], "->", new["success_rate"])

In [ ]:
# SOTA — capture a user feedback signal and route failures back into the eval set.
EVAL_SET = []
def log_feedback(question, answer, thumbs_up):
    record = {"q": question, "a": answer, "label": "good" if thumbs_up else "bad"}
    if not thumbs_up:
        EVAL_SET.append({"q": question, "expected": "<fix me>"})   # becomes a regression test
    # in prod you'd also push this to Langfuse scores / a dataset.
    return record

log_feedback("What is RAG?", "A type of cloth.", thumbs_up=False)
log_feedback("What is RAG?", "Retrieval-augmented generation.", thumbs_up=True)
print("new regression cases queued for eval:", EVAL_SET)

## 5. Benchmarks

Benchmarks are **shared, standardized eval sets** so you can compare agents/models
apples-to-apples: **SWE-bench** (fix real GitHub issues), **GAIA** (general
assistant tasks), **τ-bench** (tool-agent-user), **HELM** (holistic). The usual
headline metric is **pass@1** (fraction solved on the first attempt).

**From scratch:** a mini-benchmark runner computing pass@1 over a task suite.

> 📚 **Resources:** [SWE-bench — Jimenez et al., 2023](https://arxiv.org/abs/2310.06770) ·
> [GAIA — Mialon et al., 2023](https://arxiv.org/abs/2311.12983) ·
> [HELM](https://crfm.stanford.edu/helm/)

In [ ]:
# From scratch — a mini benchmark harness (pass@1 over a task suite).
SUITE = [
    {"id": "add",     "input": "3+5",         "check": lambda o: o == "8"},
    {"id": "upper",   "input": "say HI",      "check": lambda o: o == "HI"},
    {"id": "reverse", "input": "reverse abc", "check": lambda o: o == "cba"},
]
def system_under_test(task_input):
    if "+" in task_input: return str(eval(task_input))
    if task_input.startswith("say"): return task_input.split()[-1]
    if task_input.startswith("reverse"): return task_input.split()[-1][::-1]
    return ""

def run_benchmark(suite, sut):
    results = [{"id": t["id"], "passed": bool(t["check"](sut(t["input"])))} for t in suite]
    passed = sum(r["passed"] for r in results)
    return {"pass@1": round(passed/len(suite), 2), "passed": passed, "total": len(suite),
            "results": results}

print(json.dumps(run_benchmark(SUITE, system_under_test), indent=2))

In [ ]:
# SOTA — the same harness shape, but tasks come from a real benchmark dataset.
# Pattern (pseudo — needs `pip install datasets`, and a real agent as the SUT):
#
#   from datasets import load_dataset
#   swe = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
#   for task in swe:
#       patch = my_agent.solve(task["problem_statement"], repo=task["repo"])
#       passed = run_repo_tests(task, patch)        # apply patch, run the repo's test suite
#   # report pass@1 across all tasks
#
# Real benchmarks swap our toy check() for executing the actual project test suite
# in a sandbox (ties back to Unit 5 §6) — the metric and loop are identical.
print("See the commented pattern — real benchmarks run the project's own tests to grade.")

## Recap

Measure the agent, then see inside it:

| component | mechanism (from scratch) | SOTA packaging |
|---|---|---|
| evals | labelled set + outcome & trajectory scorers | OpenAI Evals, promptfoo, LangSmith; semantic match |
| LLM-as-judge | rubric keyword scorer | model judge w/ rubric, pointwise + pairwise |
| tracing | nested-span context manager | OpenTelemetry GenAI, Langfuse, Phoenix, LangSmith |
| monitoring | windowed metrics + drift check | Langfuse analytics, Evidently, feedback → eval set |
| benchmarks | pass@1 over a task suite | SWE-bench, GAIA, τ-bench, HELM |

The loop closes here: trace runs → eval them (outcome + trajectory, with judges
where needed) → monitor in production → feed failures back into the eval set →
benchmark against the field. The through-line across all seven units holds — each
mechanism is a few lines of plain Python, and SOTA swaps in a purpose-built tool.

That completes the core components of `index.md` (Sections 1, 3–8). Remaining
sections — **Cost/latency (§9)** and **Learning & adaptation (§10)** — follow the
same two-cell pattern if you want to continue.